# MultitaskOCTA on Kaggle — several runs in one session
Runs the presets named in `RUN_LIST` one after the other (each preset = one experiment of the project, all folds), writes one zip per run and
stops starting new folds when the time budget `MAX_HOURS` would be exceeded.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**, both datasets attached (`multitaskocta-code`, `octa500-b5-subset`).
Then edit only cell 1 (`RUN_LIST`) and use **Save Version → Save & Run All (Commit)**. Afterwards download every `<RUN_NAME>.zip` from the *Output* tab.
About 35 minutes per fold for 40 epochs on a T4: a 5-fold preset takes about 3 hours.

In [ ]:
# 1. What to run ---------------------------------------------------------------------------------
RUN_LIST = ["R3_shuffled_masks", "R4_no_segmentation_loss"]   # presets (cell 2) run one after the other, about 3 h each
SEED = None        # None = default seed of the code (2021). A number (e.g. 1) trains with that seed and appends _s<seed> to every run name
MAX_HOURS = 10.0   # no new fold is started if it would not finish within this budget (Kaggle sessions last about 9-12 h)
OVERRIDE = {}      # applied to every preset, e.g. dict(epochs=2, startpoint=1, folds=[0]) for a smoke test
LIMIT = 0          # > 0 builds only the first N scans (smoke test: LIMIT=30)
DEVICE = "cuda"    # "cpu" for a local dry run

In [ ]:
# 2. Presets (normally not edited) --------------------------------------------------------------------
BASE = ["--full_eval"]
REG = ["--strong_aug", "--label_smoothing", "0.1", "--cls_dropout", "0.3", "--weight_decay", "0.01"]   # regularisation used by the *reg runs
ALL_FOLDS = [0, 1, 2, 3, 4]

def P(name, extra=BASE, folds=ALL_FOLDS, epochs=40, startpoint=8, script="faz_vessel_seg_cls.py", fov="native"):
    return dict(name=name, extra=list(extra), folds=list(folds), epochs=epochs, startpoint=startpoint, script=script, fov=fov)

PRESETS = {
    "timing_fold0":            P("time_fold0", folds=[0]),
    "cv40":                    P("ILM_OPL_large_cv40"),                                                    # reported protocol
    "thesis100":               P("ILM_OPL_large_100", epochs=100, startpoint=20),                         # thesis length (about 7 h)
    "cv40reg":                 P("ILM_OPL_large_cv40reg", BASE + REG),
    "R2_detach_folds234":      P("ILM_OPL_large_cv40reg_detach_f234", BASE + REG + ["--detach_clf_features"], folds=[2, 3, 4]),
    "R3_shuffled_masks":       P("ILM_OPL_large_cv40reg_shufmask", BASE + REG + ["--shuffle_masks"]),
    "R4_no_segmentation_loss": P("ILM_OPL_large_cv40reg_noseg", BASE + REG + ["--seg_loss_weights", "0", "0", "0"]),
    "cls40":                   P("ILM_OPL_large_cls40", script="classification_baseline.py"),
    "cls40reg":                P("ILM_OPL_large_cls40reg", BASE + REG, script="classification_baseline.py"),
    "fov3_cv40":               P("ILM_OPL_large_fov3_cv40", fov="center3mm"),
    "fov3_cls40":              P("ILM_OPL_large_fov3_cls40", script="classification_baseline.py", fov="center3mm"),
}
unknown = [k for k in RUN_LIST if k not in PRESETS]
assert not unknown, f"unknown preset(s) {unknown}; available: {sorted(PRESETS)}"
RUNS = []
for key in RUN_LIST:
    r = dict(PRESETS[key]); r.update(OVERRIDE)
    if SEED is not None:
        r["name"] += f"_s{SEED}"
    RUNS.append(r)
for r in RUNS:
    print(f"{r['name']:42s} {r['script']:28s} folds {r['folds']}  {r['epochs']} epochs  fov {r['fov']}  {' '.join(r['extra'])}")

In [ ]:
# 3. Environment ----------------------------------------------------------------
import os, glob, shutil, subprocess, sys, json, time
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("KAGGLE_WORKING", "/kaggle/working")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ONE GPU: DataParallel on 2 GPUs would split the batch of 4 and change BatchNorm statistics
os.environ["WANDB_MODE"] = "disabled"
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

def find(pattern):
    hits = glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)
    assert hits, f"nothing matching {pattern} under {INPUT} - did you add the datasets to the notebook?"
    return hits[0]

code_dir = os.path.join(WORK, "MultitaskOCTA")
shutil.copytree(os.path.dirname(find("faz_vessel_seg_cls.py")), code_dir, dirs_exist_ok=True)
octa500 = os.path.dirname(os.path.dirname(find(os.path.join("Label", "GT_FAZ"))))
print("code:", code_dir, "| raw data:", octa500)

In [ ]:
# 4. Dependencies (torch/torchvision come with the image) ------------------------------
# Installed one by one so that a failure names the package; only the two pinned ones are essential.
reqs = [l.strip() for l in open(os.path.join(code_dir, "requirements-kaggle.txt")) if l.strip() and not l.startswith("#")]
failed = []
for r_ in reqs:
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", r_], capture_output=True, text=True)
    print(("OK    " if res.returncode == 0 else "FAILED"), r_)
    if res.returncode: failed.append(r_); print(res.stderr[-600:])
print("failed:", failed)

In [ ]:
# 5. Build the processed data sets that the selected runs need (B5 / ILM_OPL, GT_FAZ + GT_LargeVessel) ------------
data_dirs = {}
for fov in sorted({r["fov"] for r in RUNS}):
    d = os.path.join(WORK, "data", f"ILM_OPL_large_{fov}")
    cmd = [sys.executable, "Preprocessing/build_processed_octa500.py", "--octa500", octa500, "--out", d, "--projection", "ILM_OPL", "--vessel", "large", "--fov_mode", fov]
    if LIMIT: cmd += ["--limit", str(LIMIT)]
    subprocess.run(cmd, cwd=code_dir, check=True)
    data_dirs[fov] = d

In [ ]:
# 6. Train: every run, every fold, one after the other (per-fold log, timing, one zip per run) ----------------------
t_start = time.time()
min_per_epoch = None          # measured on the folds that were already trained
stopped = None
summary = []
for r in RUNS:
    out = os.path.join(WORK, "runs", r["name"]); os.makedirs(os.path.join(out, "summary"), exist_ok=True)
    done = []
    for fold in r["folds"]:
        elapsed_h = (time.time() - t_start) / 3600
        need_h = (min_per_epoch * r["epochs"] * 1.15 / 60) if min_per_epoch else 0.0
        if elapsed_h + need_h > MAX_HOURS:
            stopped = f"stopped before {r['name']} fold {fold}: {elapsed_h:.2f} h used + {need_h:.2f} h needed > MAX_HOURS = {MAX_HOURS}"
            break
        cmd = [sys.executable, r["script"], "--img_path", os.path.join(data_dirs[r["fov"]], "image_surface"), "--gt_path", os.path.join(data_dirs[r["fov"]], "gt_full", "mask"),
               "--save_path", out, "--train_type", "Vessel_FAZ", "--distance_type", "dist_signed11", "--num_epochs", str(r["epochs"]), "--startpoint", str(r["startpoint"]),
               "--fold", str(fold), "--eval_every", "3", "--num_workers", "2", "--device", DEVICE] + (["--seed", str(SEED)] if SEED is not None else []) + r["extra"]
        print(f"[{elapsed_h:5.2f} h] {r['name']} fold {fold}", flush=True)
        t0 = time.time()
        with open(os.path.join(out, f"train_fold{fold}.log"), "w") as log:
            p = subprocess.Popen(cmd, cwd=code_dir, stdout=log, stderr=subprocess.STDOUT); p.wait()
        minutes = (time.time() - t0) / 60
        print(f"          exit code {p.returncode}, {minutes:.1f} min", flush=True)
        if p.returncode == 0:
            done.append(fold); min_per_epoch = minutes / r["epochs"]
    if done:
        subprocess.run([sys.executable, "aggregate_folds.py", out], cwd=code_dir)
        shutil.make_archive(os.path.join(WORK, r["name"]), "zip", os.path.join(WORK, "runs"), r["name"])
    summary.append((r["name"], done, r["folds"]))
    if stopped:
        break
print("\n".join(f"{n}: folds finished {d} of {f}" for n, d, f in summary))
print(stopped or "all runs finished")

In [ ]:
# 7. Output files (download the zips from the Output tab) ----------------------------------------------------
for f in sorted(glob.glob(os.path.join(WORK, "*.zip"))):
    print(f"{os.path.getsize(f) / 1e6:7.0f} MB  {os.path.basename(f)}")